In [17]:
import pandas as pd
import numpy as np

# ==========================================
# STEP 1: Load the Dataset
# ==========================================
# Read the CSV file into a pandas DataFrame

file_path = r"data/cardiac_failure/labs.csv"
data = pd.read_csv(file_path)

print(f"1. Initial dataset loaded successfully. Shape: {data.shape}")

1. Initial dataset loaded successfully. Shape: (2008, 107)


**1.Quick File Reload Reference**

In [18]:
data = pd.read_csv(r"data/cardiac_failure/labs.csv")

**2.Initial Data Inspection (Head, Shape, Columns, and Tail)**

In [19]:
print(data.head())
print(data.shape)
print(data.columns)
print(data.tail())

   inpatient_number  body_temperature  pulse  respiration  \
0            857781              36.7     87           19   
1            743087              36.8     95           18   
2            866418              36.5     98           18   
3            775928              36.0     73           19   
4            810128              35.0     88           19   

   systolic_blood_pressure  diastolic_blood_pressure  map_value  fio2  \
0                      102                        64  76.666667    33   
1                      150                        70  96.666667    33   
2                      102                        67  78.666667    33   
3                      110                        74  86.000000    33   
4                      134                        62  86.000000    33   

   creatinine_enzymatic_method   urea  ...  measured_residual_base  \
0                        108.3  12.55  ...                    -2.1   
1                         62.0   4.29  ...            

**3.Checking for Duplicate Rows**

In [20]:
# 1. Count the total number of duplicate rows
duplicate_count = data.duplicated().sum()
print(f"Total duplicate rows found: {duplicate_count}")

Total duplicate rows found: 0


**4.Scanning for 100% Empty Columns**

In [21]:
# Find columns where all values are null
empty_cols = [col for col in data.columns if data[col].isnull().all()]

print(f"Total columns in dataset: {len(data.columns)}")
if empty_cols:
    print(f"100% empty columns found: {empty_cols}")
else:
    print("No 100% empty columns found.")

Total columns in dataset: 107
100% empty columns found: ['cholinesterase']


**5.Dropping Fully Empty Columns**

In [22]:
#STEP 3: Drop 100% Empty Columns
# ==========================================
# Identify columns where every single row is missing (NaN)
empty_cols = [col for col in data.columns if data[col].isnull().all()]
data = data.drop(columns=empty_cols)
print(f"3. Dropped completely empty columns: {empty_cols}")

3. Dropped completely empty columns: ['cholinesterase']


**6.Physiological Range Validation: Body Temperature**

In [23]:
# ==========================================
# Check Temperature Column
# ==========================================

# 1. View basic statistical summary (min, max, mean, percentiles)
print("=== Temperature Summary Statistics ===")
print(data['body_temperature'].describe())

# 2. Check for missing values
missing_temp = data['body_temperature'].isnull().sum()
print(f"\nMissing temperature values: {missing_temp}")

# 3. Check for physiologically extreme values 
# (Human body temperature normally ranges between ~35°C and ~42°C in clinical settings)
low_temp = data[data['body_temperature'] < 35.0]
high_temp = data[data['body_temperature'] > 42.0]

print(f"\nReadings below 35°C (Hypothermia / Extreme): {len(low_temp)}")
print(f"Readings above 42°C (Hyperthermia / Extreme): {len(high_temp)}")

# 4. Optional: View the actual extreme rows if any exist
if len(low_temp) > 0 or len(high_temp) > 0:
    print("\nExtreme temperature rows:")
    extreme_temps = pd.concat([low_temp, high_temp])
    print(extreme_temps[['inpatient_number', 'body_temperature']])
else:
    print("\nAll temperature values fall within the 35°C – 42°C clinical range.")

=== Temperature Summary Statistics ===
count    2008.000000
mean       36.416484
std         0.439529
min        35.000000
25%        36.200000
50%        36.300000
75%        36.500000
max        42.000000
Name: body_temperature, dtype: float64

Missing temperature values: 0

Readings below 35°C (Hypothermia / Extreme): 0
Readings above 42°C (Hyperthermia / Extreme): 0

All temperature values fall within the 35°C – 42°C clinical range.


**7.Generating a Missing Values Report**
**Imputing (guessing) values when 90% of the data is missing will heavily bias dataset. However, dropping them entirely means losing potential insights.So going with assumption doctor didn't order a test( sometimes the patient was stable.**

In [24]:
# ==========================================
# STEP: Analyze Missing Values
# ==========================================
import pandas as pd

# Create a summary dataframe for missing values
missing_summary = pd.DataFrame({
    'Missing Count': data.isnull().sum(),
    'Missing Percentage (%)': (data.isnull().sum() / len(data)) * 100
})

# Filter to show only columns that actually have missing values, sorted from highest to lowest
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]
missing_summary = missing_summary.sort_values(by='Missing Percentage (%)', ascending=False)

print(f"Total columns with missing values: {len(missing_summary)}")
print("\nTop columns with the highest missing rates:")
print(missing_summary.head(15).to_string())

# Optional: Save the full missing values report to a CSV file
missing_summary.to_csv('missing_values_report.csv')
print("\nFull missing values report saved to 'missing_values_report.csv'.")

Total columns with missing values: 98

Top columns with the highest missing rates:
                                Missing Count  Missing Percentage (%)
homocysteine                             1862               92.729084
apolipoprotein_a                         1832               91.235060
apolipoprotein_b                         1832               91.235060
lipoprotein                              1832               91.235060
erythrocyte_sedimentation_rate           1701               84.711155
myoglobin                                1610               80.179283
inorganic_phosphorus                     1601               79.731076
serum_magnesium                          1601               79.731076
glutamic_oxaliplatin                     1416               70.517928
high_sensitivity_protein                 1067               53.137450
reduced_hemoglobin                       1016               50.597610
methemoglobin                            1016               50.597610
hematoc

**8.Verifying Column Data Types**

In [25]:

print("=== Column Data Types ===")
print(data.dtypes)

print("\n=== Checking object columns for numeric values ===")
for col in data.select_dtypes(include=['object']).columns:
    
    try:
        sample_converted = pd.to_numeric(df[col].dropna().head(100))
        print(f"Column '{col}' is stored as text (object), but contains numbers.")
    except ValueError:
        pass

=== Column Data Types ===
inpatient_number             int64
body_temperature           float64
pulse                        int64
respiration                  int64
systolic_blood_pressure      int64
                            ...   
partial_oxygen_pressure    float64
oxyhemoglobin              float64
anion_gap                  float64
free_calcium               float64
total_hemoglobin           float64
Length: 106, dtype: object

=== Checking object columns for numeric values ===


**9.Inspection for Data Type Mismatches**

In [26]:

mismatch_report = []

for col in data.columns:
    current_type = data[col].dtype
    series_clean = data[col].dropna()
    
    if len(series_clean) == 0:
        continue
        
    # If the column is stored as text (object), check if it contains numbers
    if current_type == 'object':
        numeric_conversion = pd.to_numeric(series_clean, errors='coerce')
        num_success = numeric_conversion.notnull().sum()
        
        if num_success > 0 and num_success < len(series_clean):
            mismatch_report.append({
                'Column': col,
                'Stored As': 'object (Text)',
                'Mismatch Type': 'Mixed content (contains both text strings and numbers)'
            })
        elif num_success == len(series_clean):
            mismatch_report.append({
                'Column': col,
                'Stored As': 'object (Text)',
                'Mismatch Type': 'Entirely numeric values stored as strings'
            })


if mismatch_report:
    mismatch_df = pd.DataFrame(mismatch_report)
    print("=== Potential Data Type Mismatches Found ===")
    print(mismatch_df.to_string(index=False))
else:
    print("=== No data type mismatches detected. All text/numeric types look consistent. ===")

=== No data type mismatches detected. All text/numeric types look consistent. ===


**9.Summary Statistics Table for Numerical Features**

In [27]:

numeric_cols = data.select_dtypes(include=[np.number]).columns
cols_to_check = [col for col in numeric_cols if 'number' not in col.lower() and 'id' not in col.lower()]


summary_clean = data[cols_to_check].agg(['min', 'max', 'mean']).T.round(2)


display(summary_clean)

,min,max,mean
body_temperature,35.00,42.00,36.42
pulse,0.00,198.00,85.24
respiration,0.00,36.00,19.09
systolic_blood_pressure,0.00,252.00,131.06
diastolic_blood_pressure,0.00,146.00,76.57
...,...,...,...
partial_oxygen_pressure,20.00,255.00,108.12
oxyhemoglobin,24.30,99.10,94.94
anion_gap,-1.20,43.70,14.02
free_calcium,0.89,1.39,1.11


**10.Inspecting Frequency of Zero Values across all columns**
**Patient_id's 754892,764993 have BP 0 and 
Patient_id 773886 status is dead in hospitalization_discharge table then its valid data** 

In [28]:



numeric_cols = data.select_dtypes(include=[np.number]).columns
cols_to_check = [col for col in numeric_cols if 'number' not in col.lower() and 'id' not in col.lower()]


zero_stats = []
for col in cols_to_check:
   
    zero_count = (data[col] == 0).sum()
    total_valid = data[col].notnull().sum()
    
    if zero_count > 0:
        zero_pct = (zero_count / total_valid) * 100
        zero_stats.append({
            'Column': col, 
            'Zero Count': zero_count, 
            'Total Valid Rows': total_valid,
            'Zero Percentage (%)': round(zero_pct, 2)
        })


zero_report_df = pd.DataFrame(zero_stats)

if not zero_report_df.empty:
    display(zero_report_df.sort_values(by='Zero Percentage (%)', ascending=False))
else:
    print("No columns contain a value of 0.")

,Column,Zero Count,Total Valid Rows,Zero Percentage (%)
8,eosinophil_count,202,1981,10.20
7,eosinophil_ratio,198,1981,9.99
11,methemoglobin,87,992,8.77
12,carboxyhemoglobin,36,992,3.63
6,basophil_count,54,1981,2.73
10,high_sensitivity_troponin,51,1929,2.64
5,basophil_ratio,29,1981,1.46
9,d_dimer,4,1840,0.22
2,systolic_blood_pressure,3,2008,0.15
3,diastolic_blood_pressure,3,2008,0.15


 **11.Ensuring Consistent ID Naming Convention**

In [29]:
data = data.rename(columns={'inpatient_number':'patient_id' })

print("Column renamed successfully!")
print("Updated column names snippet:", data.columns[:5].tolist())

Column renamed successfully!
Updated column names snippet: ['patient_id', 'body_temperature', 'pulse', 'respiration', 'systolic_blood_pressure']


**12.Spotting Logic Contradictions in Blood Pressure?**

In [33]:

# Check for clinical logic contradictions: Systolic should always be > Diastolic
if 'systolic_blood_pressure' in data.columns and 'diastolic_blood_pressure' in data.columns:
    
    # Find rows where systolic is less than or equal to diastolic (excluding NaNs)
    invalid_bp = data[
        (data['systolic_blood_pressure'].notna()) & 
        (data['diastolic_blood_pressure'].notna()) & 
        (data['systolic_blood_pressure'] <= data['diastolic_blood_pressure'])
    ]
    
    print(f"⚠️ Found {len(invalid_bp)} rows where Systolic <= Diastolic.")
    
    if len(invalid_bp) > 0:
        # Display a snippet of the contradictory records
        id_col = [c for c in data.columns if 'id' in c.lower() or 'number' in c.lower()][0]
        display(invalid_bp[[id_col, 'systolic_blood_pressure', 'diastolic_blood_pressure']])
else:
    print("Blood pressure columns not found.")

⚠️ Found 5 rows where Systolic <= Diastolic.


,patient_id,systolic_blood_pressure,diastolic_blood_pressure
533,754892,0,0
611,764993,0,0
691,825901,73,95
744,838870,117,118
1831,773886,0,0


In [34]:

# Select all numeric columns (excluding Patient_Id)
numeric_cols = data.select_dtypes(include=['float64', 'int64']).columns
if 'Patient_Id' in numeric_cols:
    numeric_cols = numeric_cols.drop('Patient_Id')

# Check for values less than 0
negative_findings = {}

for col in numeric_cols:
    count = (data[col] < 0).sum()
    if count > 0:
        negative_findings[col] = count

# Display the results
if negative_findings:
    print("⚠️ Columns containing negative (invalid) values:")
    for col, count in negative_findings.items():
        print(f"  - {col}: {count} negative rows found")
else:
    print("✅ Clean! No negative values found in any numeric columns.")

⚠️ Columns containing negative (invalid) values:
  - standard_residual_base: 708 negative rows found
  - measured_residual_base: 678 negative rows found
  - anion_gap: 2 negative rows found


**13.Importing file**

In [44]:
import os


output_path = os.path.expanduser('data/cleaned/labs_cleaned.csv')


data.to_csv(output_path, index=False)
print(f"File successfully saved to: {output_path}")

File successfully saved to: data/cleaned/labs_cleaned.csv


**14.Checking for Records where Vital Biomarkers like 'systolic_blood_pressure', 'diastolic_blood_pressure', 'map_value', 'pulse', 'respiration' are 0**



In [37]:

vital_cols = ['systolic_blood_pressure', 'diastolic_blood_pressure', 'map_value', 'pulse', 'respiration']


existing_vitals = [col for col in vital_cols if col in data.columns]
zero_vitals_records = data[(data[existing_vitals] == 0).any(axis=1)]

print(f"Total patient records with at least one vital sign equal to 0: {len(zero_vitals_records)}")


display(zero_vitals_records[['patient_id'] + existing_vitals])

Total patient records with at least one vital sign equal to 0: 3


,patient_id,systolic_blood_pressure,diastolic_blood_pressure,map_value,pulse,respiration
533,754892,0,0,0.0,78,20
611,764993,0,0,0.0,124,24
1831,773886,0,0,0.0,0,0


**15.Round map_value to 2 decimal value**

In [41]:

if 'map_value' in data.columns:
    data['map_value'] = data['map_value'].round(1)
    print("✅ map_value successfully rounded to 2 decimal places.")
    
    
    display(data[['map_value']].head())
else:
    print("⚠️ 'map_value' column not found in data.")

✅ map_value successfully rounded to 2 decimal places.


,map_value
0,76.7
1,96.7
2,78.7
3,86.0
4,86.0


**Change all Fields to Decimal**

In [43]:

float_cols = data.select_dtypes(include=['float', 'float64', 'float32']).columns


data[float_cols] = data[float_cols].round(1)

print(f"✅ Successfully rounded {len(float_cols)} decimal columns to 1 decimal place!")


display(data[float_cols].head())

✅ Successfully rounded 100 decimal columns to 1 decimal place!


,body_temperature,map_value,creatinine_enzymatic_method,urea,uric_acid,glomerular_filtration_rate,cystatin,white_blood_cell,monocyte_ratio,monocyte_count,...,measured_residual_base,measured_bicarbonate,carboxyhemoglobin,body_temperature_blood_gas,oxygen_saturation,partial_oxygen_pressure,oxyhemoglobin,anion_gap,free_calcium,total_hemoglobin
0,36.7,76.7,108.3,12.6,685.0,58.6,1.3,9.4,0.1,0.8,...,-2.1,21.2,0.4,37.0,97.0,93.0,95.9,17.8,1.1,125.0
1,36.8,96.7,62.0,4.3,170.0,85.4,1.2,5.3,0.1,0.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,36.5,78.7,185.1,16.0,567.0,31.5,2.4,13.0,0.1,0.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,36.0,86.0,104.8,8.2,635.0,58.0,2.3,2.2,0.1,0.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,35.0,86.0,83.9,6.9,432.0,60.5,1.4,6.1,0.1,0.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
